# Bước 1 — Load dữ liệu Olist (CSV) vào MongoDB

Đọc 5 file CSV gốc của bộ dữ liệu Olist bằng Spark, ghi vào MongoDB thành các
collection: `orders`, `reviews`, `order_items`, `customers`, `products`.

**Thay đổi so với bản cũ:**
- Không còn hard-code path Windows cá nhân — dùng `config.py` (biến môi trường + fallback).
- Ghi vào MongoDB theo kiểu *idempotent* (ghi vào collection tạm rồi `rename`),
  để nếu chạy lại hoặc lỗi giữa chừng không làm mất dữ liệu cũ.


In [1]:
import sys
sys.path.append(".")  # để import được config.py cùng thư mục src/

from config import RAW_DIR, MONGO_URI, MONGO_DB, get_raw_path

print("RAW_DIR   =", RAW_DIR)
print("MONGO_URI =", MONGO_URI)
print("MONGO_DB  =", MONGO_DB)


RAW_DIR   = C:\Users\MY LAPTOP\PycharmProjects\Sentiment\data\raw
MONGO_URI = mongodb://localhost:27017/
MONGO_DB  = olist


## Khởi tạo Spark session

Không hard-code `JAVA_HOME`/`PYSPARK_PYTHON` — các biến này phải được set ở môi trường hệ điều hành trước khi chạy (xem `config.py`).

In [2]:
from pyspark.sql import SparkSession

# MongoDB Spark Connector 11.x là bản đầu tiên hỗ trợ Spark 4.0+ (bản 10.x chỉ
# hỗ trợ tới Spark 3.5, sẽ lỗi với pyspark 4.1.2). Spark 4 bắt buộc Scala 2.13
# nên dùng artifact "_2.13". Lần đầu chạy Spark sẽ tự tải JAR này qua Maven
# (cần mạng, JAR nhỏ chỉ vài MB).
MONGO_CONNECTOR_PACKAGE = "org.mongodb.spark:mongo-spark-connector_2.13:11.1.0"

spark = (
    SparkSession.builder
    .appName("OlistLoadToMongo")
    .config("spark.jars.packages", MONGO_CONNECTOR_PACKAGE)
    .config("spark.mongodb.write.connection.uri", MONGO_URI)
    .config("spark.sql.session.timeZone", "UTC")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
spark


## Đọc CSV

Tất cả đường dẫn lấy qua `get_data_path(...)`, không còn nối chuỗi path thủ công.

In [3]:
CSV_FILES = {
    "customers": "olist_customers_dataset.csv",
    "orders": "olist_orders_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "reviews": "olist_order_reviews_dataset.csv",
    "products": "olist_products_dataset.csv",
    "category_translation": "product_category_name_translation.csv",
}

dfs = {}
for name, filename in CSV_FILES.items():
    path = str(get_raw_path(filename))
    dfs[name] = spark.read.csv(path, header=True, inferSchema=True)
    print(f"{name:20s} -> {dfs[name].count():>7,} dòng | {path}")


customers            ->  99,441 dòng | C:\Users\MY LAPTOP\PycharmProjects\Sentiment\data\raw\olist_customers_dataset.csv
orders               ->  99,441 dòng | C:\Users\MY LAPTOP\PycharmProjects\Sentiment\data\raw\olist_orders_dataset.csv
order_items          -> 112,650 dòng | C:\Users\MY LAPTOP\PycharmProjects\Sentiment\data\raw\olist_order_items_dataset.csv
reviews              -> 104,162 dòng | C:\Users\MY LAPTOP\PycharmProjects\Sentiment\data\raw\olist_order_reviews_dataset.csv
products             ->  32,951 dòng | C:\Users\MY LAPTOP\PycharmProjects\Sentiment\data\raw\olist_products_dataset.csv
category_translation ->      71 dòng | C:\Users\MY LAPTOP\PycharmProjects\Sentiment\data\raw\product_category_name_translation.csv


## Ghi vào MongoDB (idempotent)

**Trước đây:** `collection.drop()` rồi `insert_many()` — nếu insert lỗi giữa
chừng sau khi đã drop, mất luôn dữ liệu cũ, không có cách rollback.

**Bây giờ:** ghi vào collection tạm `<name>_tmp` trước, chỉ khi insert xong
toàn bộ mới `rename` đè lên collection thật (`dropTarget=True`). Nếu có lỗi
ở bước ghi tạm, collection thật vẫn nguyên vẹn.

In [4]:
from pymongo import MongoClient
from pymongo.errors import PyMongoError

def write_collection_idempotent(spark_df, mongo_uri: str, db_name: str, collection: str):
    """Ghi 1 Spark DataFrame vào MongoDB an toàn: ghi vào collection tạm
    trước, rename đè lên collection thật khi thành công. Nếu lỗi, collection
    thật không bị ảnh hưởng."""
    tmp_name = f"{collection}_tmp"
    try:
        (
            spark_df.write
            .format("mongodb")
            .mode("overwrite")
            .option("connection.uri", mongo_uri)
            .option("database", db_name)
            .option("collection", tmp_name)
            .save()
        )
    except Exception as e:
        print(f"[LỖI] Ghi tạm '{tmp_name}' thất bại, collection '{collection}' KHÔNG bị thay đổi: {e}")
        raise

    client = MongoClient(mongo_uri)
    try:
        db = client[db_name]
        db[tmp_name].rename(collection, dropTarget=True)
        print(f"OK: {collection} đã được cập nhật ({db[collection].estimated_document_count():,} documents)")
    except PyMongoError as e:
        print(f"[LỖI] Rename '{tmp_name}' -> '{collection}' thất bại: {e}")
        raise
    finally:
        client.close()


for name, spark_df in dfs.items():
    write_collection_idempotent(spark_df, MONGO_URI, MONGO_DB, name)


OK: customers đã được cập nhật (99,441 documents)
OK: orders đã được cập nhật (99,441 documents)
OK: order_items đã được cập nhật (112,650 documents)
OK: reviews đã được cập nhật (104,162 documents)
OK: products đã được cập nhật (32,951 documents)
OK: category_translation đã được cập nhật (71 documents)


## Kiểm tra nhanh sau khi load

In [5]:
client = MongoClient(MONGO_URI)
db = client[MONGO_DB]
for name in CSV_FILES:
    print(f"{name:20s} -> {db[name].estimated_document_count():>7,} documents trong MongoDB")
client.close()


customers            ->  99,441 documents trong MongoDB
orders               ->  99,441 documents trong MongoDB
order_items          -> 112,650 documents trong MongoDB
reviews              -> 104,162 documents trong MongoDB
products             ->  32,951 documents trong MongoDB
category_translation ->      71 documents trong MongoDB
